# 03 - XGBoost
## Explainable Machine Learning for Breast Cancer Risk Prediction

Notebook ini **mandiri** (bisa dijalankan sendiri tanpa bergantung ke notebook lain) dan berfokus khusus pada satu algoritma: **XGBoost**.

Alur: Load Data -> Preprocessing -> Training + Hyperparameter Tuning -> Evaluasi -> Explainable AI (SHAP) -> Simpan hasil metrik ke `results_xgboost.csv` untuk dibandingkan di `05_Model_Comparison.ipynb`.


## 1. Import Library

In [ ]:

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, roc_curve, confusion_matrix, classification_report)

RANDOM_STATE = 42
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

print('Library dasar berhasil diimpor.')

from xgboost import XGBClassifier
import shap


## 2. Load Data & Preprocessing

In [ ]:

df = pd.read_csv('data/breast-cancer.csv')
df = df.drop(columns=['id'])

le = LabelEncoder()
df['diagnosis_encoded'] = le.fit_transform(df['diagnosis'])  # B=0, M=1
print('Mapping label:', dict(zip(le.classes_, le.transform(le.classes_))))

X = df.drop(columns=['diagnosis', 'diagnosis_encoded'])
y = df['diagnosis_encoded']
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_names, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=feature_names, index=X_test.index)

print('Train shape:', X_train.shape, '| Test shape:', X_test.shape)
df.head()


## 3. Fungsi Bantu Evaluasi

In [ ]:

def evaluate_model(model, X_te, y_te, model_name):
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    metrics = {
        'Model': model_name,
        'Accuracy': accuracy_score(y_te, y_pred),
        'Precision': precision_score(y_te, y_pred),
        'Recall': recall_score(y_te, y_pred),
        'F1-Score': f1_score(y_te, y_pred),
        'ROC-AUC': roc_auc_score(y_te, y_proba)
    }
    return metrics, y_pred, y_proba

def plot_confusion_and_roc(y_te, y_pred, y_proba, model_name, color='#3B7DD8'):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    cm = confusion_matrix(y_te, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=['Benign', 'Malignant'], yticklabels=['Benign', 'Malignant'], cbar=False)
    axes[0].set_title(f'Confusion Matrix - {model_name}', fontweight='bold')
    axes[0].set_xlabel('Prediksi')
    axes[0].set_ylabel('Aktual')

    fpr, tpr, _ = roc_curve(y_te, y_proba)
    auc = roc_auc_score(y_te, y_proba)
    axes[1].plot(fpr, tpr, color=color, linewidth=2.2, label=f'AUC = {auc:.4f}')
    axes[1].plot([0, 1], [0, 1], linestyle='--', color='gray')
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title(f'Kurva ROC - {model_name}', fontweight='bold')
    axes[1].legend(loc='lower right')

    plt.tight_layout()
    plt.show()


## 4. Training & Hyperparameter Tuning - XGBoost

Model berbasis pohon tidak memerlukan scaling fitur, sehingga kita gunakan `X_train` / `X_test` asli.


In [ ]:

xgb_param_grid = {
    'n_estimators': [200, 400],
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

xgb_grid = GridSearchCV(
    XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss'),
    xgb_param_grid, cv=cv, scoring='roc_auc', n_jobs=-1
)
xgb_grid.fit(X_train, y_train)

model = xgb_grid.best_estimator_
print('Best XGBoost params:', xgb_grid.best_params_)
print('Best CV ROC-AUC:', round(xgb_grid.best_score_, 4))


## 5. Evaluasi Model - XGBoost

In [ ]:

metrics, y_pred, y_proba = evaluate_model(model, X_test, y_test, 'XGBoost')
metrics_df = pd.DataFrame([metrics]).set_index('Model').round(4)
metrics_df


In [ ]:

plot_confusion_and_roc(y_test, y_pred, y_proba, 'XGBoost', color='#D64545')


In [ ]:

print(classification_report(y_test, y_pred, target_names=['Benign', 'Malignant']))


## 6. Simpan Hasil Metrik (results_xgboost.csv)

In [ ]:

metrics_df.to_csv('results_xgboost.csv')
print('Hasil metrik disimpan ke results_xgboost.csv')
metrics_df


## 7. Explainable AI (XAI) - XGBoost

### 7.1 Feature Importance Bawaan XGBoost

In [ ]:

importance = pd.Series(model.feature_importances_, index=feature_names).sort_values(ascending=False)
plt.figure(figsize=(8, 7))
importance.head(15).sort_values().plot(kind='barh', color='#D64545')
plt.title('Top 15 Feature Importance - XGBoost', fontweight='bold')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()


### 7.2 SHAP Summary Plot

In [ ]:

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test, show=False)
plt.title('SHAP Summary Plot - XGBoost', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### 7.3 SHAP Waterfall Plot - Interpretasi 1 Pasien

In [ ]:

idx = 0
sample_pred = model.predict(X_test.iloc[[idx]])[0]
sample_actual = y_test.iloc[idx]
print(f"Prediksi: {'Malignant' if sample_pred==1 else 'Benign'} | Aktual: {'Malignant' if sample_actual==1 else 'Benign'}")

shap_explanation = shap.Explanation(
    values=shap_values[idx],
    base_values=explainer.expected_value,
    data=X_test.iloc[idx],
    feature_names=feature_names
)

shap.plots.waterfall(shap_explanation, show=False)
plt.title(f'SHAP Waterfall - Contoh Pasien #{idx} (XGBoost)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## 8. Kesimpulan Singkat - XGBoost

- Ringkasan metrik model ini tersimpan di `results_xgboost.csv` dan akan digabungkan dengan hasil model lain di `05_Model_Comparison.ipynb`.
- Model beserta parameter terbaik hasil `GridSearchCV` ditampilkan pada bagian training di atas.
- Interpretasi SHAP menunjukkan fitur-fitur mana yang paling berkontribusi terhadap prediksi model ini.
